# Gemini-oefeningen — gratis API-tier

In dit notebook oefen je met de Gemini API via Google AI Studio. Deze variant gebruikt **geen Vertex AI, GCP-project of Cloud Storage-bucket**.

> **Vooraf:** Maak in [Google AI Studio](https://aistudio.google.com/apikey) een eigen Gemini API-key aan. Kies geen betaald plan en koppel geen billing-account. De gratis tier heeft gebruikslimieten en de beschikbaarheid kan per account/model veranderen. Deel je key niet en gebruik geen gevoelige of persoonlijke data. Bij gratis gebruik kan Google prompts en antwoorden gebruiken om producten te verbeteren; zie de actuele [voorwaarden](https://ai.google.dev/gemini-api/terms).

Als Google AI Studio vraagt om billing of je account geen gratis toegang heeft, stop dan en laat het de docent weten.

In [ ]:
%pip install -q -U google-genai requests pillow

In [ ]:
from getpass import getpass
from google import genai
from google.genai import types
import json

# Voer je eigen key in. getpass verbergt de invoer en slaat de key niet op in deze cel.
GEMINI_API_KEY = getpass('Plak je Gemini API-key (invoer blijft verborgen): ')
client = genai.Client(api_key=GEMINI_API_KEY)

# Gebruik een snel, efficiënt model. Controleer in AI Studio welke modellen
# op dit moment beschikbaar zijn binnen jouw gratis tier.
GEMINI_MODEL = 'gemini-3.1-flash-lite'
print('Client ingesteld voor', GEMINI_MODEL)

---
## Oefening 1 — Je eerste Gemini API-call

Een SDK (Software Development Kit) is een bibliotheek waarmee Python op een vaste manier met een API kan praten. Voer de voorbeeldcall uit.

In [ ]:
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents='Leg in één zin uit wat cloud computing is.',
    config=types.GenerateContentConfig(max_output_tokens=120),
)
print(response.text)

### Jouw beurt
Pas de vraag aan. Wat voor antwoord krijg je als je Gemini vraagt om de uitleg aan een eerstejaarsstudent te geven?

In [ ]:
prompt = 'VUL HIER JE INSTRUCTIE/VRAAG IN'
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=150),
)
print(response.text)

---
## Oefening 2 — Variatie in output

De temperatuur beïnvloedt hoe gevarieerd de woordkeuze kan zijn. Een lagere temperatuur geeft vaak consistentere antwoorden; een hogere temperatuur kan meer variatie geven. Dit maakt output niet automatisch beter of betrouwbaarder. Vergelijk bijvoorbeeld 0.2 en 1.0 met dezelfde prompt.

In [ ]:
config = types.GenerateContentConfig(temperature=1.0, max_output_tokens=120)
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents='Bedenk een korte slogan voor een energiezuinig datacenter.',
    config=config,
)
print(response.text)

# Opdracht: verander temperature naar 1.0 en vergelijk de uitkomst.

# Opdracht: verander de prompt en experimenteer verder met de temperature setting.

## Oefening 3: Structured output

Je kunt het model forceren een antwoord te geven in JSON in een bepaald format, door:
1. Expliciet in de prompt te vragen om JSON output met een specifiek schema
2. Een voorbeeld te geven van het gewenste formaat (=*1-shot learning*)

**Opdracht:**

Tweek de prompt totdat je een valide JSON hebt

In [ ]:
prompt = '''
Analyseer Google Cloud Run.
service_name (string), category (string), pricing_model (string), 
primary_use_cases (lijst van drie strings), pros (lijst van twee strings), cons (lijst van twee strings), difficulty_level (integer van 1 t/m 10).
'''
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=500,
        # response_mime_type='application/json',
    ),
)
print(response.text)

try:
    parsed = json.loads(response.text)
    print('Valide JSON ontvangen!')
    print('Service:', parsed['service_name'])
    print('Categorie:', parsed['category'])
except (json.JSONDecodeError, KeyError) as error:
    print('Geen valide JSON, controleer de velden:', error)

---
## Oefening 4 — Een afbeelding analyseren

Gemini kan tekst en afbeeldingen in één verzoek verwerken. Dit voorbeeld haalt een openbare afbeelding op en stuurt de bytes mee. Gebruik alleen afbeeldingen die je mag delen met de API.

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from IPython.display import display

image_url = 'https://www.google.com/images/branding/googlelogo/2x/googlelogo_color_272x92dp.png'
image_response = requests.get(image_url, timeout=20)
image_response.raise_for_status()
image_bytes = image_response.content
image = Image.open(BytesIO(image_bytes))
display(image)

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        'Beschrijf deze afbeelding kort. Welk bedrijf of product zie je?',
        types.Part.from_bytes(data=image_bytes, mime_type='image/png'),
    ],
    config=types.GenerateContentConfig(max_output_tokens=120),
)
print(response.text)

### Jouw beurt
Pas `image_url` aan naar een publiek beschikbare afbeelding. Laat Gemini een gerichte vraag over die afbeelding beantwoorden.

In [ ]:
# Jouw code hier - kies een image_url van een publieke afbeelding die je mag delen.
my_image_url = 'https://...'
my_question = ''


---
## Oefening 5 — Sensorgegevens als lokaal bestand

In een echte productie omgeving zou je toegang hebben tot GCP Cloud Storage-bucket, maar hier gebruiken we kleine voorbeeldbestanden in de notebookomgeving. Daarmee oefenen we JSON lezen zonder Cloud Storage of billing.

In [ ]:
from pathlib import Path

sensor_day1 = [
    {'city': 'Rotterdam', 'sensor_id': 'RDM-01', 'temperature_c': 19.8, 'co2_ppm': 810},
    {'city': 'Utrecht', 'sensor_id': 'UTR-03', 'temperature_c': 21.1, 'co2_ppm': 1240},
    {'city': 'Delft', 'sensor_id': 'DLF-02', 'temperature_c': 20.4, 'co2_ppm': 960},
]
sensor_day2 = [
    {'city': 'Rotterdam', 'sensor_id': 'RDM-01', 'temperature_c': 20.0, 'co2_ppm': 845},
    {'city': 'Utrecht', 'sensor_id': 'UTR-03', 'temperature_c': 21.4, 'co2_ppm': 1510},
    {'city': 'Delft', 'sensor_id': 'DLF-02', 'temperature_c': 20.1, 'co2_ppm': 920},
]

Path('sensor_readings_day1.json').write_text(json.dumps(sensor_day1, indent=2), encoding='utf-8')
Path('sensor_readings_day2.json').write_text(json.dumps(sensor_day2, indent=2), encoding='utf-8')

sensor_data = json.loads(Path('sensor_readings_day1.json').read_text(encoding='utf-8'))
print(f'{len(sensor_data)} metingen geladen')
print(json.dumps(sensor_data[0], indent=2, ensure_ascii=False))

### Jouw beurt
Lees `sensor_readings_day2.json` in en vergelijk de CO₂-metingen met dag 1. Welke locatie verdient nader onderzoek?

In [ ]:
# Implementeer hier: laad dag 2 en toon bijvoorbeeld de locaties met co2_ppm > 1000.

---
## Oefening 6 — Data analyse met Gemini

Stuur een kleine selectie van de voorbeelddata als context naar Gemini en geef het een Data Analyse vraag, vergelijk dit met jouw eigen implementatie van oefening 5. 

In [ ]:
sensor_text = json.dumps(sensor_data, ensure_ascii=False) # <<---- zo wordt een dict als een string geconverteerd
prompt = (
    'JOUW PROMPT HIER\n\n' # <<---- vervang dit
    f'Metingen: {sensor_text}'# <<---- zo wordt een variabele in de prompt gezet
)
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=180),
)
print(response.text)

## Afronding

- Welke taak loste Gemini goed op, en welke moest je zelf controleren?
- Wat veranderde toen je de prompt of temperatuur aanpaste?
- Welke beperkingen merkte je van een API en een gratis gebruikslimiet?
